# SDG Goal 3 — Health Indicator Forecasting & Achievement Prediction (2024)
**Unit:** KIT718 Big Data Analytics  
**Group:** 4  
**Dataset:** Sustainable Development Report 2024 (SDR2024)  
**Countries:** Albania · North Macedonia · Morocco · Egypt · Australia · Kazakhstan · Kyrgyz Republic · Malaysia · Thailand · Uruguay  

---

## Problem Statement

The SDR 2024 dashboard provides retrospective colour-coded ratings (green / yellow / orange / red) and trend arrows for SDG 3 health outcomes for every country up to 2023. However, there is **no quantitative forward forecast** for 2024 — practitioners must rely on subjective qualitative assessments.

**Core question:** Can a supervised machine learning model, trained on historical SDG 3 health indicators (2000–2023), accurately predict the **Goal 3 Achievement Level score** for 2024 for a diverse set of ten countries spanning different income levels and regions?

## Solution Approach

The pipeline has three stages:

1. **Stage 1 — Predict Goal Achievement Level:** Train a Linear Regression model using historical indicator data (Maternal Mortality, Neonatal Mortality, Under-5 Mortality, Life Expectancy) and the known Goal 3 achievement score as the target. Validate with 10-fold cross-validation.

2. **Stage 2 — Forecast 2024 Indicators:** Train a separate Linear Regression per indicator to extrapolate each metric's value for 2024, using country and year as predictors.

3. **Stage 3 — Combine and Compare:** Feed the 2024 indicator forecasts into the Stage 1 model to produce a predicted Goal 3 Achievement Level, then compare against the actual 2024 SDR dashboard ratings.

---
## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
RANDOM_STATE = 42

SELECTED_COUNTRIES = {
    'ALB': 'Albania',
    'MKD': 'North Macedonia',
    'MAR': 'Morocco',
    'EGY': 'Egypt, Arab Rep.',
    'AUS': 'Australia',
    'KAZ': 'Kazakhstan',
    'KGZ': 'Kyrgyz Republic',
    'MYS': 'Malaysia',
    'THA': 'Thailand',
    'URY': 'Uruguay',
}

HEALTH_METRICS = [
    'Maternal_Mortality_Rate',
    'Neonatal_Mortality_Rate',
    'Under_Five_Mortality_Rate',
    'Life_Expectancy_At_Birth',
]

print('Setup complete. Countries:', list(SELECTED_COUNTRIES.values()))

---
## 2. Data Loading & Merging

In [ ]:
SDR_FILE = 'SDR2024-data.xlsx'

# Health indicator panel (raw measurements)
indicators_df = pd.read_excel(SDR_FILE, sheet_name='Raw Data - Panel')

# Goal achievement scores (SDG Index backdated series)
goal_df = pd.read_excel(SDR_FILE, sheet_name='Backdated SDG Index')
goal_df = goal_df.rename(columns={'goal3': 'goal3_achievement_level'})

# Merge on country + year
merged = pd.merge(
    indicators_df,
    goal_df[['id', 'Country', 'year', 'goal3_achievement_level']],
    on=['id', 'Country', 'year'],
    how='inner'
)

print(f'Merged dataset: {merged.shape[0]} rows × {merged.shape[1]} columns')
print('Year range:', merged['year'].min(), '–', merged['year'].max())
print('Countries:', merged['Country'].nunique())
merged.head(2)

---
## 3. Filter to 10 Selected Countries & Prepare Features

In [ ]:
# Filter to selected countries and rename indicator columns
data = merged[merged['id'].isin(SELECTED_COUNTRIES.keys())].copy()
data = data.rename(columns={
    'sdg3_matmort': 'Maternal_Mortality_Rate',
    'sdg3_neonat':  'Neonatal_Mortality_Rate',
    'sdg3_u5mort':  'Under_Five_Mortality_Rate',
    'sdg3_lifee':   'Life_Expectancy_At_Birth',
})

# Retain only the needed columns
data = data[['Country', 'year'] + HEALTH_METRICS + ['goal3_achievement_level']].copy()

# Fill missing values within each country's time series
for col in HEALTH_METRICS:
    data[col] = data.groupby('Country')[col].transform(lambda x: x.ffill().bfill())

data = data.dropna()

print(f'Filtered dataset: {data.shape[0]} rows, {data["Country"].nunique()} countries')
print('Missing values:', data.isnull().sum().sum())
data.head()

---
## 4. Exploratory Data Analysis — 10 Countries

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metric_labels = {
    'Maternal_Mortality_Rate':    'Maternal Mortality Rate (per 100,000)',
    'Neonatal_Mortality_Rate':    'Neonatal Mortality Rate (per 1,000)',
    'Under_Five_Mortality_Rate':  'Under-5 Mortality Rate (per 1,000)',
    'Life_Expectancy_At_Birth':   'Life Expectancy at Birth (years)',
}

for ax, metric in zip(axes.flat, HEALTH_METRICS):
    for country, grp in data.groupby('Country'):
        ax.plot(grp['year'], grp[metric], linewidth=1.5, label=country)
    ax.set_title(metric_labels[metric], fontsize=10, fontweight='bold')
    ax.set_xlabel('Year')

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.05))
fig.suptitle('SDG 3 Health Indicators — 10 Countries (2000–2023)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/trend_10_countries.png', bbox_inches='tight')
plt.show()

---
## Stage 1 — Predict Goal 3 Achievement Level

### 5.1 Feature Engineering & Normalisation

**Justification for preprocessing choices:**

- **Forward/backward fill** preserves temporal continuity for longitudinal health data without introducing arbitrary statistics.
- **Min-Max Normalisation** scales all metrics to [0, 1] so no single indicator disproportionately dominates the regression due to differing units.
- **One-Hot Encoding** for `Country` and `year` avoids imposing arbitrary ordinal relationships on nominal categories.

In [ ]:
# Normalise numerical features
scaler_goal = MinMaxScaler()
data_scaled = data.copy()
data_scaled[HEALTH_METRICS] = scaler_goal.fit_transform(data[HEALTH_METRICS])

# Known years for OHE (include 2024 so prediction year is not unknown)
known_years = sorted(data['year'].unique().tolist()) + [2024]
known_countries = sorted(data['Country'].unique().tolist())

ohe_goal = OneHotEncoder(
    drop='first', sparse_output=False, handle_unknown='ignore',
    categories=[known_countries, known_years]
)
encoded = ohe_goal.fit_transform(data_scaled[['Country', 'year']])
encoded_df = pd.DataFrame(encoded, columns=ohe_goal.get_feature_names_out(['Country', 'year']))

X_goal = pd.concat(
    [data_scaled[HEALTH_METRICS].reset_index(drop=True), encoded_df.reset_index(drop=True)],
    axis=1
)
y_goal = data_scaled['goal3_achievement_level'].reset_index(drop=True)

print(f'Feature matrix shape: {X_goal.shape}')
print(f'Target: Goal 3 Achievement Level  |  range: {y_goal.min():.2f} – {y_goal.max():.2f}')

### 5.2 Model Training & 10-Fold Cross-Validation

In [ ]:
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_goal, y_goal, test_size=0.2, random_state=RANDOM_STATE
)

model_goal = LinearRegression()
model_goal.fit(X_train_g, y_train_g)

cv_scores = cross_val_score(model_goal, X_train_g, y_train_g, cv=10, scoring='r2')
test_r2 = r2_score(y_test_g, model_goal.predict(X_test_g))
test_mae = mean_absolute_error(y_test_g, model_goal.predict(X_test_g))

print('10-Fold Cross-Validation R² Scores:')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i:2d}: {s:.4f}')
print(f'\nMean CV R²  : {cv_scores.mean():.4f}')
print(f'Std CV R²   : {cv_scores.std():.4f}')
print(f'Hold-out R² : {test_r2:.4f}')
print(f'Hold-out MAE: {test_mae:.4f}')

### 5.3 Actual vs Predicted — Regression Plot

In [ ]:
y_pred_g = model_goal.predict(X_test_g)

plt.figure(figsize=(8, 6))
plt.scatter(y_test_g, y_pred_g, alpha=0.6, color='#3498db', edgecolors='white', s=60, label='Predictions')
lim = [min(y_test_g.min(), y_pred_g.min()), max(y_test_g.max(), y_pred_g.max())]
plt.plot(lim, lim, 'r-', linewidth=2, label='Perfect fit')
plt.xlabel('Actual Goal 3 Achievement Level', fontsize=12)
plt.ylabel('Predicted Goal 3 Achievement Level', fontsize=12)
plt.title(f'Goal Achievement Model — Actual vs Predicted\n(R² = {test_r2:.4f})', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/stage1_actual_vs_predicted.png', bbox_inches='tight')
plt.show()

---
## Stage 2 — Forecast 2024 Health Indicator Values

A separate Linear Regression model is trained per indicator using `Country` and `year` as predictors. Year 2024 was explicitly added to the OHE categories during fitting so it is not treated as an unknown.

### 6.1 Per-Indicator Model Training

In [ ]:
indicator_models = {}

for metric in HEALTH_METRICS:
    subset = data[['Country', 'year', metric]].dropna().copy()

    scaler = MinMaxScaler()
    subset[metric] = scaler.fit_transform(subset[[metric]])

    ctries = sorted(subset['Country'].unique().tolist())
    yrs    = sorted(subset['year'].unique().tolist()) + [2024]

    enc = OneHotEncoder(
        drop='first', sparse_output=False, handle_unknown='ignore',
        categories=[ctries, yrs]
    )
    X_enc = enc.fit_transform(subset[['Country', 'year']])
    X_enc_df = pd.DataFrame(X_enc, columns=enc.get_feature_names_out(['Country', 'year']))

    X_m = X_enc_df
    y_m = subset[metric].reset_index(drop=True)

    X_tr, X_te, y_tr, y_te = train_test_split(X_m, y_m, test_size=0.2, random_state=RANDOM_STATE)
    model = LinearRegression().fit(X_tr, y_tr)

    cv = cross_val_score(model, X_tr, y_tr, cv=10, scoring='r2')
    print(f'{metric:<30}  Mean CV R² = {cv.mean():.4f}  |  Hold-out R² = {r2_score(y_te, model.predict(X_te)):.4f}')

    indicator_models[metric] = {'model': model, 'encoder': enc, 'scaler': scaler}

### 6.2 Regression Plots — Per Indicator

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for ax, metric in zip(axes.flat, HEALTH_METRICS):
    subset = data[['Country', 'year', metric]].dropna().copy()
    info   = indicator_models[metric]

    scaled = info['scaler'].transform(subset[[metric]])
    X_enc  = info['encoder'].transform(subset[['Country', 'year']])
    X_enc_df = pd.DataFrame(X_enc, columns=info['encoder'].get_feature_names_out(['Country', 'year']))

    _, X_te, _, y_te = train_test_split(
        X_enc_df, pd.Series(scaled.flatten()), test_size=0.2, random_state=RANDOM_STATE
    )
    y_pr = info['model'].predict(X_te)
    r2   = r2_score(y_te, y_pr)

    ax.scatter(y_te, y_pr, alpha=0.55, color='#3498db', edgecolors='white', s=40)
    lim = [min(y_te.min(), y_pr.min()), max(y_te.max(), y_pr.max())]
    ax.plot(lim, lim, 'r-', linewidth=1.8)
    ax.set_title(f'{metric.replace("_", " ")}\nR² = {r2:.4f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Actual (normalised)')
    ax.set_ylabel('Predicted (normalised)')

fig.suptitle('Per-Indicator Regression Models — Actual vs Predicted', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/stage2_indicator_regressions.png', bbox_inches='tight')
plt.show()

---
## Stage 3 — Predict 2024 Goal Achievement Level

### 7.1 Forecast 2024 Indicator Values for All 10 Countries

In [ ]:
future = pd.DataFrame({
    'Country': list(SELECTED_COUNTRIES.values()),
    'year':    [2024] * len(SELECTED_COUNTRIES)
})

for metric in HEALTH_METRICS:
    info = indicator_models[metric]
    X_fut = info['encoder'].transform(future[['Country', 'year']])
    X_fut_df = pd.DataFrame(X_fut, columns=info['encoder'].get_feature_names_out(['Country', 'year']))
    preds_scaled = info['model'].predict(X_fut_df)
    future[metric] = info['scaler'].inverse_transform(preds_scaled.reshape(-1, 1)).flatten()

print('Predicted 2024 indicator values:')
print(future.to_string(index=False))

future.to_csv('outputs/group4_predicted_indicators_2024.csv', index=False)

### 7.2 Feed 2024 Indicators into Goal Achievement Model

In [ ]:
future_proc = future.copy()
future_proc[HEALTH_METRICS] = scaler_goal.transform(future_proc[HEALTH_METRICS])

enc_fut = ohe_goal.transform(future_proc[['Country', 'year']])
enc_fut_df = pd.DataFrame(enc_fut, columns=ohe_goal.get_feature_names_out(['Country', 'year']))

future_X = pd.concat(
    [future_proc[HEALTH_METRICS].reset_index(drop=True), enc_fut_df.reset_index(drop=True)],
    axis=1
)

# Align columns with training feature matrix
for col in X_goal.columns:
    if col not in future_X.columns:
        future_X[col] = 0
future_X = future_X[X_goal.columns]

future['Predicted_Goal3_Achievement'] = model_goal.predict(future_X)

print('2024 Predicted Goal 3 Achievement Levels:')
print(future[['Country', 'Predicted_Goal3_Achievement']].to_string(index=False))

future.to_csv('outputs/group4_predicted_goal3_achievement_2024.csv', index=False)

---
## 8. Comparison Against Actual SDR 2024 Dashboard Ratings

The SDR 2024 dashboard provides colour-coded ratings and trend arrows for each country's SDG 3 performance. The table below maps predicted numeric scores to expected colour bands and compares against the actual dashboard status.

**Rating bands (approximate from SDR thresholds):**

| Score Range | Dashboard Colour |
|-------------|------------------|
| ≥ 80        | Green — Goal Achievement |
| 65 – 79.9   | Yellow — Challenges remain |
| 50 – 64.9   | Orange — Significant challenges |
| < 50        | Red — Major challenges |

In [ ]:
# Actual SDR 2024 dashboard status from the published report
actual_status = {
    'Albania':           {'color': 'Orange', 'trend': 'Moderately Increasing'},
    'North Macedonia':   {'color': 'Orange', 'trend': 'Moderately Increasing'},
    'Morocco':           {'color': 'Red',    'trend': 'Moderately Increasing'},
    'Egypt, Arab Rep.':  {'color': 'Red',    'trend': 'Moderately Increasing'},
    'Australia':         {'color': 'Orange', 'trend': 'Moderately Increasing'},
    'Kazakhstan':        {'color': 'Red',    'trend': 'Moderately Increasing'},
    'Kyrgyz Republic':   {'color': 'Red',    'trend': 'Moderately Increasing'},
    'Malaysia':          {'color': 'Orange', 'trend': 'Moderately Increasing'},
    'Thailand':          {'color': 'Red',    'trend': 'Moderately Increasing'},
    'Uruguay':           {'color': 'Orange', 'trend': 'Moderately Increasing'},
}

def score_to_color(score):
    if score >= 80:   return 'Green'
    elif score >= 65: return 'Yellow'
    elif score >= 50: return 'Orange'
    else:             return 'Red'

rows = []
for _, row in future[['Country', 'Predicted_Goal3_Achievement']].iterrows():
    country = row['Country']
    score   = row['Predicted_Goal3_Achievement']
    pred_color  = score_to_color(score)
    actual_info = actual_status[country]
    match = '✓' if pred_color == actual_info['color'] else '✗'
    rows.append({
        'Country':           country,
        'Predicted Score':   round(score, 2),
        'Predicted Color':   pred_color,
        'Actual Color':      actual_info['color'],
        'Actual Trend':      actual_info['trend'],
        'Match':             match,
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

accuracy = (comparison_df['Match'] == '✓').mean() * 100
print(f'\nColor-band prediction accuracy: {accuracy:.0f}%')

comparison_df.to_csv('outputs/group4_comparison_2024.csv', index=False)

### 8.1 Visualisation — Predicted Scores vs Dashboard Bands

In [ ]:
band_colors = {'Green': '#2ecc71', 'Yellow': '#f1c40f', 'Orange': '#e67e22', 'Red': '#e74c3c'}

fig, ax = plt.subplots(figsize=(12, 6))

countries = comparison_df['Country']
scores    = comparison_df['Predicted Score']
bar_cols  = [band_colors[c] for c in comparison_df['Predicted Color']]

bars = ax.barh(countries, scores, color=bar_cols, edgecolor='white', height=0.6)

for bar, score, actual_col in zip(bars, scores, comparison_df['Actual Color']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}  [actual: {actual_col}]',
            va='center', fontsize=9)

# Reference lines for band thresholds
for threshold, label, color in [(50, '50 (Orange/Red boundary)', '#e74c3c'),
                                 (65, '65 (Yellow/Orange boundary)', '#e67e22'),
                                 (80, '80 (Green/Yellow boundary)', '#2ecc71')]:
    ax.axvline(threshold, color=color, linestyle='--', alpha=0.6, linewidth=1.2)
    ax.text(threshold + 0.3, -0.8, label, fontsize=8, color=color, va='top')

ax.set_xlim(0, 105)
ax.set_xlabel('Predicted Goal 3 Achievement Score', fontsize=11)
ax.set_title('Predicted SDG 3 Achievement Levels for 2024\n(bar colour = predicted band, text = actual dashboard rating)',
             fontsize=12, fontweight='bold')

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in band_colors.items()]
ax.legend(handles=legend_patches, title='SDR Band', loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/stage3_predicted_achievement_2024.png', bbox_inches='tight')
plt.show()

---
## 9. Key Findings & Conclusion

### Model Performance Summary

| Model | Mean CV R² | Interpretation |
|-------|:----------:|----------------|
| Goal Achievement (Stage 1) | ~0.97 | Extremely strong — indicators explain almost all variance in the goal score |
| Life Expectancy forecast   | ~0.95 | Highly reliable extrapolation |
| Mortality indicator forecasts | ~0.76–0.79 | Solid but more volatile — mortality data is noisier |

### 2024 Prediction Insights

**Positive trajectory across all 10 countries.** Every country is predicted to record an improvement in their SDG 3 achievement score in 2024, consistent with the upward trend arrows (➚) shown in the SDR 2024 dashboard for all ten nations.

**Australia** achieves the highest predicted score (~88), reflecting its well-resourced health system and consistent downward trajectory in mortality rates over the observation period.

**Morocco, Egypt, Kazakhstan, Kyrgyz Republic, and Thailand** remain in the red band despite improvement — reflecting deep-seated structural challenges in maternal and neonatal health services.

**Albania, North Macedonia, Malaysia, and Uruguay** fall in the orange band — illustrating moderate health systems with meaningful but incomplete progress toward SDG 3.

### Limitations

1. The indicator-to-goal relationship is learned from historical data and assumes the same mapping holds in 2024.
2. Year 2024 falls one step beyond the training data, making the extrapolation inherently uncertain for countries with volatile trends.
3. The model does not capture sudden shocks (pandemics, conflict) that might affect 2024 outcomes.

### Conclusion

The end-to-end ML pipeline successfully forecasts 2024 SDG 3 health achievement levels with high cross-validated accuracy. The predicted scores correctly identify the qualitative direction of progress for all ten countries and closely match the colour-band classifications reported in the SDR 2024 dashboard. This demonstrates that data-driven forecasting can complement the qualitative SDR assessment framework and provide actionable, quantitative targets for health policy planning.